In [0]:
%sql
-- 1. Criando o catálogo
CREATE CATALOG IF NOT EXISTS proj_dw;
USE CATALOG proj_dw;

-- 2. Criando os schemas
CREATE SCHEMA IF NOT EXISTS landing_manual;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS dw_star;
CREATE SCHEMA IF NOT EXISTS dw_snowflake;

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%sql
-- Com o arquivo sales_order dentro do volume ja criado no schema landing
-- Criando nossa tabela base, a partir dos dados da landing
-- Three level namespace (catalog_name.schema_name.object_name)

CREATE OR REPLACE TABLE proj_dw.landing_manual.sales_orders_raw AS
SELECT *
FROM read_files(
  '/Volumes/proj_dw/landing_manual/input/sales_orders.csv',
  header => 'true',
  inferSchema => 'true'
)

In [0]:
%sql
SELECT *
FROM proj_dw.landing_manual.sales_orders_raw
LIMIT 5

## Criacao da tabela bronze

- Como estou criando uma tabela vinda de um CSV, é muito importante explicitar o schema dessa tabela
- É bem comum o spark induzir errado o tipo do dado de alguma coluna principalmente se for string

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.bronze.sales_orders AS
SELECT
  CAST(order_id AS STRING) AS order_id,
  CAST(order_item_id AS INT) AS order_item_id,
  CAST(order_date AS DATE) AS order_date,
  CAST(shipment_date AS DATE) AS shipment_date,
  CAST(customer_id AS STRING) AS customer_id,
  customer_name,
  customer_email,
  CAST(customer_birthdate AS DATE) AS customer_birthdate,
  customer_gender,
  customer_city,
  customer_state,
  CAST(product_id AS STRING) AS product_id,
  product_name,
  product_category,
  product_subcategory,
  brand,
  CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
  CAST(quantity AS INT) AS quantity,
  CAST(discount_value AS DECIMAL(10,2)) AS discount_value,
  CAST(store_id AS STRING) AS store_id,
  store_name,
  store_city,
  store_state,
  payment_method,
  CAST(payment_installments AS INT) AS payment_installments,
  payment_status,
  -- métricas calculadas
  (unit_price * quantity) AS total_gross,
  (unit_price * quantity) - discount_value AS total_net

FROM proj_dw.landing_manual.sales_orders_raw;

In [0]:
%sql
SELECT *
FROM proj_dw.bronze.sales_orders
LIMIT 10